In [102]:
# some useful mysklearn package import statements and reloads
import importlib

import myutils
importlib.reload(myutils)
import myutils as myutils

# uncomment once you paste your mypytable.py into mysklearn package
import mypytable
importlib.reload(mypytable)
from mypytable import MyPyTable 

# uncomment once you paste your myclassifiers.py into mysklearn package
import myclassifiers
importlib.reload(myclassifiers)

import myevaluation
importlib.reload(myevaluation)
import myevaluation as myevaluation

from tabulate import tabulate

# MBTI PERSONALITY CLASSIFIER

Authors: *Xavier Barinaga & Sam Allen*

---

### <u> Our Goal</u>


Classify a user by their personality type, using their latest 50 tweets.


### <u> Our Data</u> 

#### The Personality Types

The 4 personality attributes we are trying to predict based of the tweets are:

 * `Introversion` (I) *vs.* `Extroversion (E)`
 * `Intuition (N)` *vs.* `Sensing (S)`
 * `Thinking (T)` *vs.* `Feeling (F)`
 * `Judging (J)` *vs.* `Perceiving (P)`


#### The Features:

We extracted the following features from the tweets in preprocessing:

`Word Count` - Total Words in the tweets

`Polarity` - Measures emotional tone. | **-1.0 (negative) to +1.0 (positive)**  

`Subjectivity` - Measures how personal, opinionated, or factual the text is. | **0.0 (objective) to 1.0 (subjective)**  

`TTR` - Measures word uniqueness | $\frac{\text{Unique Words}}{\text{Total Words}} $

`Adjective Ratio` - Measures Adjective Ratio |  $\frac{\text{Total Adjectives}}{\text{Total Words}} $

`Verb Ratio` - Measures Verb Ratio |  $\frac{\text{Total Adverbs}}{\text{Total Words}} $

`Personal Pronoun Ratio` - Measures Personal Pronoun Ratio |  $\frac{\text{Total Personal Pronouns}}{\text{Total Words}} $

`Third-Person Pronoun Ratio` - Measures Third-Person Pronoun Ratio |  $\frac{\text{Total Third-Person Pronouns}}{\text{Total Words}} $

***~For each instance (user) we are averaging the attribute values across all their tweets.***

### <u>Cleaning </u> 

Our data started out with large imbalances. So we decided for each of the 4 target attributes, we will even the distribution before training the model.

Our extracted features came as very specific floating point numbers. We have only implemented categorical implementations of our classification so we had to discretize our extracted features.

Our tweets were also seperated by "|||" so we had to break apart the tweets by splitting on this. 

There is a certain point at which we have to accept that natural language is messy, especially tweets. This means we can't go through each tweet individually to personally see if we think the tweet is useful or not. For example, some people might just tweet a link to a youtube video. Something like that is just part of processing tweets.

### <u> Classifying </u>

#### kNN

In [103]:
mbti = MyPyTable()
mbti.load_from_file("categorical.csv")

ie = []
ns = []
tf = []
jp = []

# Discretize each numeric column
mbti = myutils.discretize(mbti, 10)

# Separate each unique class label
for row in mbti.data:
    ie.append(row[0][0]) # append I for introversion or E for extroversion
    ns.append(row[0][1]) # append N for Intution or S for sensing
    tf.append(row[0][2]) # append T for Thinking or F for feeling
    jp.append(row[0][3]) # append J for Judging or P for perceiving

# Drop all columns except the most informative ones
mbti.drop_column("type")
# mbti.drop_column("Word Count")
# mbti.drop_column("Polarity")
# mbti.drop_column("TTR")
# mbti.drop_column("Subjectivity")
mbti.drop_column("Verb Ratio")
mbti.drop_column("Adjective Ratio")
mbti.drop_column("Personal Pronoun Ratio")
mbti.drop_column("Third-Person Pronoun Ratio")

new_data_ie, new_column_ie = mbti.even_class_distribution(ie)
new_data_ns, new_column_ns = mbti.even_class_distribution(ns)
new_data_tf, new_column_tf = mbti.even_class_distribution(tf)
new_data_jp, new_column_jp = mbti.even_class_distribution(jp)

In [104]:
print(
"""
===========================================
            Total Instances
===========================================
""")
print("Introvert vs Extrovert Classifier:", len(new_column_ie))
print("ntuition vs Sensing Classifier:", len(new_column_ns))
print("Thinking vs Feeling Classifier:", len(new_column_tf))
print("Judging vs Perceiving Classifier:", len(new_column_jp))


            Total Instances

Introvert vs Extrovert Classifier: 3998
ntuition vs Sensing Classifier: 2394
Thinking vs Feeling Classifier: 7962
Judging vs Perceiving Classifier: 6868


In [105]:
X_ie = new_data_ie.data
y_ie = new_column_ie
X_ns = new_data_ns.data
y_ns = new_column_ns
X_tf = new_data_tf.data
y_tf = new_column_tf
X_jp = new_data_jp.data
y_jp = new_column_jp

## kNN Classifier
Below we will explore the accuracy of our kNN algorithm for each of our personality types. In general, our kNN algorithm performed the best using 51 neighbors.

In [106]:
knn_ie = myclassifiers.MyKNeighborsClassifier(n_neighbors=51)
X_train_ie, X_test_ie, y_train_ie, y_test_ie = myevaluation.stratified_split(X_ie, y_ie, test_size=0.33)

knn_ns = myclassifiers.MyKNeighborsClassifier(n_neighbors=51)
X_train_ns, X_test_ns, y_train_ns, y_test_ns = myevaluation.stratified_split(X_ns, y_ns, test_size=0.33)

knn_tf = myclassifiers.MyKNeighborsClassifier(n_neighbors=51)
X_train_tf, X_test_tf, y_train_tf, y_test_tf = myevaluation.stratified_split(X_tf, y_tf, test_size=0.33)

knn_jp = myclassifiers.MyKNeighborsClassifier(n_neighbors=51)
X_train_jp, X_test_jp, y_train_jp, y_test_jp = myevaluation.stratified_split(X_jp, y_jp, test_size=0.33)

In [107]:
knn_ie.fit(X_train_ie, y_train_ie)
y_pred_ie = knn_ie.predict(X_test_ie)

knn_ns.fit(X_train_ns, y_train_ns)
y_pred_ns = knn_ns.predict(X_test_ns)

knn_tf.fit(X_train_tf, y_train_tf)
y_pred_tf = knn_tf.predict(X_test_tf)

knn_jp.fit(X_train_jp, y_train_jp)
y_pred_jp = knn_jp.predict(X_test_jp)

In [108]:
print(
"""
===========================================
          kNN Predictive Accuracy
===========================================
"""
f"kNN Classifier for Introversion: Accuracy = {round(myevaluation.accuracy_score(y_test_ie, y_pred_ie), 2)}, F1 Score: {round(myevaluation.binary_f1_score(y_test_ie, y_pred_ie, ['I', 'E']), 2)}\n"
f"kNN Classifier for Intuition: Accuracy = {round(myevaluation.accuracy_score(y_test_ns, y_pred_ns), 2)}, F1 Score: {round(myevaluation.binary_f1_score(y_test_ns, y_pred_ns, ['N', 'S']), 2)}\n"
f"kNN Classifier for Judging: Accuracy = {round(myevaluation.accuracy_score(y_test_tf, y_pred_tf), 2)}, F1 Score: {round(myevaluation.binary_f1_score(y_test_tf, y_pred_tf, ['T', 'F']), 2)}\n"
f"kNN Classifier for Thinking: Accuracy = {round(myevaluation.accuracy_score(y_test_jp, y_pred_jp), 2)}, F1 Score: {round(myevaluation.binary_f1_score(y_test_jp, y_pred_jp, ['J', 'P']), 2)} \n"
)


          kNN Predictive Accuracy
kNN Classifier for Introversion: Accuracy = 0.53, F1 Score: 0.57
kNN Classifier for Intuition: Accuracy = 0.55, F1 Score: 0.52
kNN Classifier for Judging: Accuracy = 0.59, F1 Score: 0.61
kNN Classifier for Thinking: Accuracy = 0.5, F1 Score: 0.5 



In [109]:
header = ["", "Introvert", "Extrovert", "Total", "Recognition (%)"]
print(tabulate(myutils.clean_cf_matrix(myevaluation.confusion_matrix(y_test_ie, y_pred_ie, ['I', 'E']), ['Introvert', 'Extrovert']), headers=header))

             Introvert    Extrovert    Total  Recognition (%)
---------  -----------  -----------  -------  -----------------
Introvert          403          256      659  61.15%
Extrovert          363          296      659  44.92%


In [110]:
header = ["", "Intuition", "Sensing", "Total", "Recognition (%)"]
print(tabulate(myutils.clean_cf_matrix(myevaluation.confusion_matrix(y_test_ns, y_pred_ns, ['N', 'S']), ['Intuition', 'Sensing']), headers=header))

             Intuition    Sensing    Total  Recognition (%)
---------  -----------  ---------  -------  -----------------
Intuition          191        204      395  48.35%
Sensing            150        245      395  62.03%


In [111]:
header = ["", "Thinking", "Feeling", "Total", "Recognition (%)"]
print(tabulate(myutils.clean_cf_matrix(myevaluation.confusion_matrix(y_test_tf, y_pred_tf, ['T', 'F']), ['Thinking', 'Feeling']), headers=header))

            Thinking    Feeling    Total  Recognition (%)
--------  ----------  ---------  -------  -----------------
Thinking         858        455     1313  65.35%
Feeling          632        681     1313  51.87%


In [112]:
header = ["", "Judging", "Perceiving", "Total", "Recognition (%)"]
print(tabulate(myutils.clean_cf_matrix(myevaluation.confusion_matrix(y_test_jp, y_pred_jp, ['J', 'P']), ['Judging', 'Perceiving']), headers=header))

              Judging    Perceiving    Total  Recognition (%)
----------  ---------  ------------  -------  -----------------
Judging           555           578     1133  48.98%
Perceiving        546           587     1133  51.81%


### Decision Tree

In [113]:
dt_ie = myclassifiers.MyDecisionTreeClassifier()
X_train_ie, X_test_ie, y_train_ie, y_test_ie = myevaluation.stratified_split(X_ie, y_ie, test_size=0.33)

dt_ns = myclassifiers.MyDecisionTreeClassifier()
X_train_ns, X_test_ns, y_train_ns, y_test_ns = myevaluation.stratified_split(X_ns, y_ns, test_size=0.33)

dt_tf = myclassifiers.MyDecisionTreeClassifier()
X_train_tf, X_test_tf, y_train_tf, y_test_tf = myevaluation.stratified_split(X_tf, y_tf, test_size=0.33)

dt_jp = myclassifiers.MyDecisionTreeClassifier()
X_train_jp, X_test_jp, y_train_jp, y_test_jp = myevaluation.stratified_split(X_jp, y_jp, test_size=0.33)

In [114]:
dt_ie.fit(X_train_ie, y_train_ie)
y_pred_ie = dt_ie.predict(X_test_ie)

dt_ns.fit(X_train_ns, y_train_ns)
y_pred_ns = dt_ns.predict(X_test_ns)

dt_tf.fit(X_train_tf, y_train_tf)
y_pred_tf = dt_tf.predict(X_test_tf)

dt_jp.fit(X_train_jp, y_train_jp)
y_pred_jp = dt_jp.predict(X_test_jp)

In [115]:
print(
"""
===========================================
     Decision Tree Predictive Accuracy     
===========================================
"""
f"Decision Tree Classifier for Introversion: Accuracy = {round(myevaluation.accuracy_score(y_test_ie, y_pred_ie), 2)}, F1 Score: {round(myevaluation.binary_f1_score(y_test_ie, y_pred_ie, ['I', 'E']), 2)}\n"
f"Decision Tree Classifier for Intuition: Accuracy = {round(myevaluation.accuracy_score(y_test_ns, y_pred_ns), 2)}, F1 Score: {round(myevaluation.binary_f1_score(y_test_ns, y_pred_ns, ['N', 'S']), 2)}\n"
f"Decision Tree Classifier for Judging: Accuracy = {round(myevaluation.accuracy_score(y_test_tf, y_pred_tf), 2)}, F1 Score: {round(myevaluation.binary_f1_score(y_test_tf, y_pred_tf, ['T', 'F']), 2)}\n"
f"Decision Tree Classifier for Thinking: Accuracy = {round(myevaluation.accuracy_score(y_test_jp, y_pred_jp), 2)}, F1 Score: {round(myevaluation.binary_f1_score(y_test_jp, y_pred_jp, ['J', 'P']), 2)} \n"
)


     Decision Tree Predictive Accuracy     
Decision Tree Classifier for Introversion: Accuracy = 0.52, F1 Score: 0.48
Decision Tree Classifier for Intuition: Accuracy = 0.52, F1 Score: 0.55
Decision Tree Classifier for Judging: Accuracy = 0.58, F1 Score: 0.59
Decision Tree Classifier for Thinking: Accuracy = 0.51, F1 Score: 0.53 



In [116]:
header = ["", "Introvert", "Extrovert", "Total", "Recognition (%)"]
print(tabulate(myutils.clean_cf_matrix(myevaluation.confusion_matrix(y_test_ie, y_pred_ie, ['I', 'E']), ['Introvert', 'Extrovert']), headers=header))

             Introvert    Extrovert    Total  Recognition (%)
---------  -----------  -----------  -------  -----------------
Introvert          291          368      659  44.16%
Extrovert          262          397      659  60.24%


In [117]:
header = ["", "Intuition", "Sensing", "Total", "Recognition (%)"]
print(tabulate(myutils.clean_cf_matrix(myevaluation.confusion_matrix(y_test_ns, y_pred_ns, ['N', 'S']), ['Intuition', 'Sensing']), headers=header))

             Intuition    Sensing    Total  Recognition (%)
---------  -----------  ---------  -------  -----------------
Intuition          232        163      395  58.73%
Sensing            214        181      395  45.82%


In [118]:
header = ["", "Thinking", "Feeling", "Total", "Recognition (%)"]
print(tabulate(myutils.clean_cf_matrix(myevaluation.confusion_matrix(y_test_tf, y_pred_tf, ['T', 'F']), ['Thinking', 'Feeling']), headers=header))

            Thinking    Feeling    Total  Recognition (%)
--------  ----------  ---------  -------  -----------------
Thinking         780        533     1313  59.41%
Feeling          569        744     1313  56.66%


In [119]:
header = ["", "Judging", "Perceiving", "Total", "Recognition (%)"]
print(tabulate(myutils.clean_cf_matrix(myevaluation.confusion_matrix(y_test_jp, y_pred_jp, ['J', 'P']), ['Judging', 'Perceiving']), headers=header))

              Judging    Perceiving    Total  Recognition (%)
----------  ---------  ------------  -------  -----------------
Judging           631           502     1133  55.69%
Perceiving        602           531     1133  46.87%


### Random Forest

In [133]:
rf_ie = myclassifiers.MyRandomForestClassifier(n_estimators=50, n_features=0.6, m=25)
X_train_ie, X_test_ie, y_train_ie, y_test_ie = myevaluation.stratified_split(X_ie, y_ie, test_size=0.33)

rf_ns = myclassifiers.MyRandomForestClassifier(n_estimators=50, n_features=0.6, m=25)
X_train_ns, X_test_ns, y_train_ns, y_test_ns = myevaluation.stratified_split(X_ns, y_ns, test_size=0.33)

rf_tf = myclassifiers.MyRandomForestClassifier(n_estimators=50, n_features=0.6, m=25)
X_train_tf, X_test_tf, y_train_tf, y_test_tf = myevaluation.stratified_split(X_tf, y_tf, test_size=0.33)

rf_jp = myclassifiers.MyRandomForestClassifier(n_estimators=50, n_features=0.6, m=25)
X_train_jp, X_test_jp, y_train_jp, y_test_jp = myevaluation.stratified_split(X_jp, y_jp, test_size=0.33)

In [134]:
rf_ie.fit(X_train_ie, y_train_ie)
y_pred_ie = rf_ie.predict(X_test_ie)

rf_ns.fit(X_train_ns, y_train_ns)
y_pred_ns = rf_ns.predict(X_test_ns)

rf_tf.fit(X_train_tf, y_train_tf)
y_pred_tf = rf_tf.predict(X_test_tf)

rf_jp.fit(X_train_jp, y_train_jp)
y_pred_jp = rf_jp.predict(X_test_jp)

In [136]:
print(
"""
===========================================
     Random Forest Predictive Accuracy
===========================================
"""
f"Random Forest Classifier for Introversion: Accuracy = {round(myevaluation.accuracy_score(y_test_ie, y_pred_ie), 2)}, F1 Score: {round(myevaluation.binary_f1_score(y_test_ie, y_pred_ie, ['I', 'E']), 2)}\n"
f"Random Forest Classifier for Intuition: Accuracy = {round(myevaluation.accuracy_score(y_test_ns, y_pred_ns), 2)}, F1 Score: {round(myevaluation.binary_f1_score(y_test_ns, y_pred_ns, ['N', 'S']), 2)}\n"
f"Random Forest Classifier for Judging: Accuracy = {round(myevaluation.accuracy_score(y_test_tf, y_pred_tf), 2)}, F1 Score: {round(myevaluation.binary_f1_score(y_test_tf, y_pred_tf, ['T', 'F']), 2)}\n"
f"Random Forest Classifier for Thinking: Accuracy = {round(myevaluation.accuracy_score(y_test_jp, y_pred_jp), 2)}, F1 Score: {round(myevaluation.binary_f1_score(y_test_jp, y_pred_jp, ['J', 'P']), 2)} \n"
)


     Random Forest Predictive Accuracy
Random Forest Classifier for Introversion: Accuracy = 0.55, F1 Score: 0.56
Random Forest Classifier for Intuition: Accuracy = 0.55, F1 Score: 0.61
Random Forest Classifier for Judging: Accuracy = 0.59, F1 Score: 0.61
Random Forest Classifier for Thinking: Accuracy = 0.52, F1 Score: 0.57 



In [137]:
header = ["", "Introvert", "Extrovert", "Total", "Recognition (%)"]
print(tabulate(myutils.clean_cf_matrix(myevaluation.confusion_matrix(y_test_ie, y_pred_ie, ['I', 'E']), ['Introvert', 'Extrovert']), headers=header))

             Introvert    Extrovert    Total  Recognition (%)
---------  -----------  -----------  -------  -----------------
Introvert          381          278      659  57.81%
Extrovert          319          340      659  51.59%


In [138]:
header = ["", "Intuition", "Sensing", "Total", "Recognition (%)"]
print(tabulate(myutils.clean_cf_matrix(myevaluation.confusion_matrix(y_test_ns, y_pred_ns, ['N', 'S']), ['Intuition', 'Sensing']), headers=header))

             Intuition    Sensing    Total  Recognition (%)
---------  -----------  ---------  -------  -----------------
Intuition          271        124      395  68.61%
Sensing            228        167      395  42.28%


In [139]:
header = ["", "Thinking", "Feeling", "Total", "Recognition (%)"]
print(tabulate(myutils.clean_cf_matrix(myevaluation.confusion_matrix(y_test_tf, y_pred_tf, ['T', 'F']), ['Thinking', 'Feeling']), headers=header))

            Thinking    Feeling    Total  Recognition (%)
--------  ----------  ---------  -------  -----------------
Thinking         843        470     1313  64.20%
Feeling          606        707     1313  53.85%


In [140]:
header = ["", "Judging", "Perceiving", "Total", "Recognition (%)"]
print(tabulate(myutils.clean_cf_matrix(myevaluation.confusion_matrix(y_test_jp, y_pred_jp, ['J', 'P']), ['Judging', 'Perceiving']), headers=header))

              Judging    Perceiving    Total  Recognition (%)
----------  ---------  ------------  -------  -----------------
Judging           718           415     1133  63.37%
Perceiving        679           454     1133  40.07%


## Random Forest Results
We tried a wide variety of hyperparameters and found the best results using the following hyperparameters:
 * n_estimators = 50
 * m = 25
 * n_features = 0.6

Overall, we find an average accuracy of about 55% across our four binary classifiers. This result indicates to us that our features are not very indicative of personality type. We also believe that personalities may be something that is more of a figment of our imagination than something hard-coded into our biology. While it could easily be that we just did not mine the right features, it could be an indicator for some lack of realism regarding personality types
